# 📓 IGEM - Interactive Usage Examples

This notebook demonstrates **interactive usage examples** of the IGEM package, including data Load, Describe, Modify, Analyse and Plot steps.

---

## ⚙️ Requirements and Installation

To run this notebook locally, follow these steps:

```bash
$ python -m venv .venv
$ source .venv/bin/activate  # or .venv\Scripts\activate on Windows
$ pip install igem jupyter notebook
$ jupyter notebook
```
---

## 📁 Example Data
To run the examples, you will need to download the example data files available at:

👉 Download Example Files

Make sure to place the downloaded files in a folder accessible to the notebook (e.g., data/).


In [1]:
import pandas as pd
from pathlib import Path
import igem

rpy2 ModuleSpec(name='rpy2', loader=None, submodule_search_locations=_NamespacePath(['/Users/andrerico/Works/Temp/ge/venv/lib/python3.11/site-packages/rpy2']))


In [2]:
data_dir = Path.cwd() / "data"
print(data_dir)

/Users/andrerico/Works/Temp/ge/data


<h2>LOAD</h2>

<h4>LOAD PLINK FILES IN LAZY MODE</h4>

In [3]:
prefix = data_dir / "chr11"
bed = prefix.with_suffix(".bed")
bim = prefix.with_suffix(".bim")
fam = prefix.with_suffix(".fam")

In [4]:
# Load data
G = igem.load.load_plink(str(bed), str(bim), str(fam))

In [5]:
print(G.ndim)
print(set(G.dims) == {"sample", "variant"})
G

2
True


<xarray.DataArray 'genotype' (sample: 14, variant: 779)> Size: 44kB
dask.array<transpose, shape=(14, 779), dtype=float32, chunksize=(14, 779), chunktype=numpy.ndarray>
Coordinates: (12/14)
  * sample   (sample) <U4 224B 'B001' 'B002' 'B003' ... 'B012' 'B013' 'B014'
  * variant  (variant) <U10 31kB 'variant0' 'variant1' ... 'variant778'
    fid      (sample) <U4 224B 'B001' 'B002' 'B003' ... 'B012' 'B013' 'B014'
    iid      (sample) <U4 224B 'B001' 'B002' 'B003' ... 'B012' 'B013' 'B014'
    father   (sample) <U1 56B '0' '0' '0' '0' '0' '0' ... '0' '0' '0' '0' '0'
    mother   (sample) <U1 56B '0' '0' '0' '0' '0' '0' ... '0' '0' '0' '0' '0'
    ...       ...
    chrom    (variant) <U2 6kB '11' '11' '11' '11' '11' ... '11' '11' '11' '11'
    snp      (variant) <U9 28kB '316849996' '316874359' ... '345698259'
    cm       (variant) float64 6kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    pos      (variant) int64 6kB 157439 181802 248969 ... 28961091 29005702
    a0       (variant) <U1 3kB 'C' 'G' 'G' 'C' 'C' 'T' ... 'A' 'C' 'A' 'A' 'T'
    a1       (variant) <U1 3kB 'T' 'C' 'C' 'T' 'T' 'A' ... 'G' 'T' 'G' 'C' 'C'

In [6]:
variant_id = "variant5"
sample_id = "B001"
print(G.sel(sample=sample_id, variant=variant_id).values)
print(G.a0.sel(variant=variant_id).values)
print(len(G.sel(variant=variant_id).values))

0.0
T
14


<h4>LOAD TSV FILES</h4>

In [7]:
df_csv = igem.load.from_csv(data_dir / "load_one_row.csv", index_col=None)
print(df_csv)

Loaded 1 observations of 3 variables
   outcome  variable  covariant
ID                             
0     case         1          2


In [8]:
df_tsv = igem.load.from_tsv(data_dir / "load_one_row.tsv", index_col=None)
print(df_tsv)

Loaded 1 observations of 1 variables
   outcome variable    covariant
ID                              
0                  case    1   2


<h2>DESCRIBE</h2>

<h4>Describe Correlations</h4>

In [14]:
import pandas as pd
from pathlib import Path
from statsmodels import datasets

import igem

In [15]:
data_dir = Path.cwd() / "data"
print(data_dir)

/Users/andrerico/Works/Temp/ge/data


In [16]:
data = datasets.get_rdataset("plantTraits", "cluster", cache=True).data
data.index.name = "ID"

In [17]:
igem.describe.correlations(data, threshold=0.9)

,var1,var2,correlation


<h4>Describe Percent NA</h4>

In [18]:
result_perc_na = igem.describe.percent_na(data)
correct_result = pd.DataFrame(
    [
        ["pdias", 26.470588],
        ["longindex", 18.382353],
        ["durflow", 0],
        ["height", 0],
        ["begflow", 0],
    ],
    columns=["Variable", "percent_na"],
)
pd.testing.assert_frame_equal(result_perc_na.head(), correct_result)

<h4>Describe Skewness</h4>

In [19]:
dropna = True
result_skew = igem.describe.skewness(data, dropna)
if dropna:
    correct_result = pd.DataFrame(
        [
            ["pdias", "continuous", 5.156840, 9.610278, 7.235317e-22],
            ["longindex", "continuous", 0.163848, 0.742076, 4.580411e-01],
            ["durflow", "continuous", 2.754286, 8.183515, 2.756827e-16],
            ["height", "continuous", 0.583514, 2.735605, 6.226567e-03],
            ["begflow", "continuous", -0.316648, -1.549449, 1.212738e-01],
        ],
        columns=["Variable", "type", "skew", "zscore", "pvalue"],
    )
else:
    correct_result = pd.DataFrame(
        [
            ["pdias", "continuous", None, None, None],
            ["longindex", "continuous", None, None, None],
            ["durflow", "continuous", 2.754286, 8.183515, 2.756827e-16],
            ["height", "continuous", 0.583514, 2.735605, 6.226567e-03],
            ["begflow", "continuous", -0.316648, -1.549449, 1.212738e-01],
        ],
        columns=["Variable", "type", "skew", "zscore", "pvalue"],
    )
pd.testing.assert_frame_equal(result_skew.head(), correct_result)

<h2>MODIFY</h2>

In [21]:
import pandas as pd
from pathlib import Path
from statsmodels import datasets

import igem

In [22]:
data = datasets.get_rdataset("plantTraits", "cluster", cache=True).data
data.index.name = "ID"

<h4>Modify Make</h4>

In [23]:
cols = ["piq", "ros", "leafy", "winan", "suman"]

result_binary = igem.modify.make_binary(data, only=cols)
result_category = igem.modify.make_categorical(data)
result_continuous = igem.modify.make_continuous(data)
assert all(result_binary[cols].dtypes == "category")

Running make_binary
--------------------------------------------------------------------------------
Set 5 of 31 variable(s) as binary, each with 136 observations
Running make_categorical
--------------------------------------------------------------------------------
Set 31 of 31 variable(s) as categorical, each with 136 observations
Running make_continuous
--------------------------------------------------------------------------------
Set 31 of 31 variable(s) as continuous, each with 136 observations


/Users/andrerico/Works/Temp/ge/venv/lib/python3.11/site-packages/clarite/modules/modify.py:542: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  lambda col: pd.to_numeric(col, errors="ignore")


Warning fixed in next version

<h4> Modify Merge </h4>

In [26]:
df1 = data.loc[:, list(data)[:3]]
df2 = data.loc[:, list(data)[3:6]]
df3 = data.loc[:, list(data)[6:]]
df = igem.modify.merge_variables(df1, df2)
df = igem.modify.merge_variables(df, df3)
assert all(df == data)

Running merge_variables
--------------------------------------------------------------------------------
outer Merge:
	left = 136 observations of 3 variables
	right = 136 observations of 3 variables
Kept 136 observations of 6 variables.
Running merge_variables
--------------------------------------------------------------------------------
outer Merge:
	left = 136 observations of 6 variables
	right = 136 observations of 25 variables
Kept 136 observations of 31 variables.


<h4>Modify Column Filter Percent Zero</h4>

In [29]:
result_perc_zero = igem.modify.colfilter_percent_zero(data)

Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 31 of 31 continuous variables
	Removed 7 (22.58%) tested continuous variables which were equal to zero in at least 90.00% of non-NA observations.


<h4>Modify Column Filter Min N</h4>

In [39]:
n = len(data)
data_min_n = data.copy()
data_min_n["test"] = [None] + [True] * 2 + [False] * (n - 3)
data_min_n = igem.modify.make_binary(data=data_min_n, only=["test"])
result_min_n = igem.modify.colfilter_min_n(data_min_n, n=n)
assert result_min_n.shape == (136, 12)

Running make_binary
--------------------------------------------------------------------------------
Set 1 of 32 variable(s) as binary, each with 136 observations
Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 1 of 1 binary variables
	Removed 1 (100.00%) tested binary variables which had less than 136 non-null values.
Testing 0 of 0 categorical variables
Testing 31 of 31 continuous variables
	Removed 19 (61.29%) tested continuous variables which had less than 136 non-null values.


<h4>Modify Column Filter Min Cat N</h4>

In [42]:
data_cat_n = data.copy()
data_cat_n["test"] = (["cat1"] * 2 + ["cat2"] * 6 + ["cat3"] * (len(data_cat_n) - 8))
data_cat_n["test2"] = (["cat1"] * 3 + ["cat2"] * 6 + ["cat3"] * (len(data_cat_n) - 9))
data_cat_n = igem.modify.make_categorical(
        data=data_cat_n, only=["test", "test2"]
    )
result_cat_n = igem.modify.colfilter_min_cat_n(data_cat_n, n=3)
assert result_cat_n.shape == (136, 32)

Running make_categorical
--------------------------------------------------------------------------------
Set 2 of 33 variable(s) as categorical, each with 136 observations
Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 0 of 0 binary variables
Testing 2 of 2 categorical variables
	Removed 1 (50.00%) tested categorical variables which had a category with less than 3 values.


<h4>Modify Filter Incomplete Obs</h4>

In [43]:
col_names = list(data)[:4]
data_incomp = data.copy()
data_incomp.iloc[0, 0] = None
data_incomp.iloc[2, 5:7] = None
result_incomp = igem.modify.rowfilter_incomplete_obs(data_incomp, only=col_names)

assert result_incomp.shape == (91, 31)

Running rowfilter_incomplete_obs
--------------------------------------------------------------------------------
Removed 45 of 136 observations (33.09%) due to NA values in any of 4 variables


<h4>Modify Remove Outliers Gaussian</h4>

In [44]:
result_rem_outliers = igem.modify.remove_outliers(data, method="gaussian", skip=["durflow"])

assert (result_rem_outliers.isna().sum()["longindex"] == data.isna().sum()["longindex"])
assert (result_rem_outliers.isna().sum()["durflow"] == data.isna().sum()["durflow"])
assert (result_rem_outliers.isna().sum()["vegaer"] == data.isna().sum()["vegaer"] + 12)

Running remove_outliers
--------------------------------------------------------------------------------
Removing outliers from 136 observations of 30 continuous variables with values more than 3 standard deviations from the mean
	Removed 0 low and 2 high outliers from pdias (outside -589.85 to 731.38)
	Removed 0 low and 0 high outliers from longindex (outside -0.60 to 1.41)
	Removed 0 low and 0 high outliers from height (outside -2.03 to 10.26)
	Removed 0 low and 0 high outliers from begflow (outside 0.92 to 9.42)
	Removed 0 low and 0 high outliers from mycor (outside -0.42 to 3.61)
	Removed 0 low and 12 high outliers from vegaer (outside -1.56 to 1.99)
	Removed 0 low and 0 high outliers from vegsout (outside -1.76 to 2.61)
	Removed 0 low and 0 high outliers from autopoll (outside -2.49 to 4.27)
	Removed 0 low and 0 high outliers from insects (outside -2.49 to 6.53)
	Removed 0 low and 0 high outliers from wind (outside -3.70 to 5.68)
	Removed 0 low and 0 high outliers from lign (outsi

<h4>Modify Remove Outliers IQR</h4>

In [46]:
result_rem_iqr = igem.modify.remove_outliers(data, method="iqr", cutoff=1.5, skip=["durflow"])

assert (result_rem_iqr.isna().sum()["longindex"] == data.isna().sum()["longindex"])
assert (result_rem_iqr.isna().sum()["durflow"] == data.isna().sum()["durflow"]) 
assert (result_rem_iqr.isna().sum()["vegaer"] == data.isna().sum()["vegaer"] + 17)

Running remove_outliers
--------------------------------------------------------------------------------
Removing outliers from 136 observations of 30 continuous variables with values < 1st Quartile - (1.5 * IQR) or > 3rd quartile + (1.5 * IQR)
	Removed 0 low and 23 high outliers from pdias (outside -11.61 to 20.57)
	Removed 0 low and 0 high outliers from longindex (outside -1.08 to 1.81)
	Removed 0 low and 0 high outliers from height (outside -2.50 to 9.50)
	Removed 0 low and 0 high outliers from begflow (outside 1.00 to 9.00)
	Removed 0 low and 0 high outliers from mycor (outside -0.50 to 3.50)
	Removed 0 low and 17 high outliers from vegaer (outside 0.00 to 0.00)
	Removed 0 low and 0 high outliers from vegsout (outside -1.50 to 2.50)
	Removed 0 low and 0 high outliers from autopoll (outside -3.00 to 5.00)
	Removed 0 low and 0 high outliers from insects (outside -4.50 to 7.50)
	Removed 0 low and 0 high outliers from wind (outside -4.50 to 7.50)
	Removed 0 low and 0 high outliers from

<h4>Modify Transform</h4>

In [49]:
df_transform = pd.DataFrame(
    {
        "a": [10, 100, 1000],
        "b": [100, 1000, 10000],
        "c": [True, False, True],
    }
)

result_transform = igem.modify.transform(df_transform, "log10", skip=["c"])

assert all(result_transform["a"] == [1, 2, 3])
assert all(result_transform["b"] == [2, 3, 4])
assert all(result_transform["c"] == [True, False, True])

Running transform
--------------------------------------------------------------------------------
Transformed 'a' using 'log10'
Transformed 'b' using 'log10'


<h4>Modify Categorize Many String</h4>

In [51]:
df_categ = pd.DataFrame(
    {
        "a": range(100),
        "b": range(100),
        "c": [str(n) + "ABC" for n in range(100)],
    }
)
categorized = igem.modify.categorize(df_categ)
# Dtypes and data shouldn't have actually changed.  'c' will remain an 'unknown' type.
assert (categorized.dtypes == df_categ.dtypes).all()
assert (categorized == df_categ).all().all()

Running categorize
--------------------------------------------------------------------------------
0 of 3 variables (0.00%) are classified as constant (1 unique value).
0 of 3 variables (0.00%) are classified as binary (2 unique values).
0 of 3 variables (0.00%) are classified as categorical (3 to 6 unique values).
2 of 3 variables (66.67%) are classified as continuous (>= 15 unique values).
0 of 3 variables (0.00%) were dropped.
1 of 3 variables (33.33%) were not categorized and need to be set manually.
	0 variables had between 6 and 15 unique values
	1 variables had >= 15 values but couldn't be converted to continuous (numeric) values


<h2>ANALYSE</h2>

<h4>Analyse Interaction Study</h4>

<h5>NHANES Dataset</h5>

A data frame with 8591 observations on the following 7 variables.
- SDMVPSU - Primary sampling units
- SDMVSTRA - Sampling strata
- WTMEC2YR - Sampling weights
- HI_CHOL - Binary: 1 for total cholesterol over 240mg/dl, 0 under 240mg/dl
- race - Categorical (1=Hispanic, 2=non-Hispanic white, 3=non-Hispanic black, 4=other)  # noqa E501
- agecat  - Categorical Age group(0,19] (19,39] (39,59] (59,Inf]
- RIAGENDR - Binary: Gender: 1=male, 2=female


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

import igem

rpy2 ModuleSpec(name='rpy2', loader=None, submodule_search_locations=_NamespacePath(['/Users/andrerico/Works/Temp/ge/venv/lib/python3.11/site-packages/rpy2']))


In [33]:
print(data_dir)

/Users/andrerico/Works/Temp/ge/data
Loaded 8,591 observations of 7 variables


<h5>Test the nhanes dataset a specific interaction</h5>

---

In [34]:
# Prepare the data
nhanes_data = igem.load.from_csv(data_dir / "nhanes_data.csv", index_col=None)
df = igem.modify.make_binary(nhanes_data, only=["HI_CHOL", "RIAGENDR"])
df = igem.modify.make_categorical(df, only=["race", "agecat"])
df = igem.modify.colfilter(df, only=["HI_CHOL", "RIAGENDR", "race", "agecat"])

Running make_binary
--------------------------------------------------------------------------------
Set 2 of 7 variable(s) as binary, each with 8,591 observations
Running make_categorical
--------------------------------------------------------------------------------
Set 2 of 7 variable(s) as categorical, each with 8,591 observations
Running colfilter
--------------------------------------------------------------------------------
Keeping 4 of 7 variables:
	2 of 2 binary variables
	2 of 2 categorical variables
	0 of 3 continuous variables
	0 of 0 unknown variables


In [62]:
# Run Interaction without reporting betas
python_result_nobeta = igem.analyze.interaction_study(
    outcomes="HI_CHOL",
    covariates=["race"],
    data=df,
    interactions=[("agecat", "RIAGENDR")],
    report_betas=False,
)

# Run Interaction with reporting betas
python_result = igem.analyze.interaction_study(
    outcomes="HI_CHOL",
    covariates=["race"],
    data=df,
    interactions=[("agecat", "RIAGENDR")],
    report_betas=True,
)

InteractionRegression
-------------------------
Binary Outcome (family = Binomial): 'HI_CHOL'
	7,059 occurrences of '0.0' coded as 0
	787 occurrences of '1.0' coded as 1
Using 7,846 of 8,591 observations
	745 are missing a value for the outcome variable
Regressing 3 variables
	1 binary variables
	1 categorical variables
	1 continuous variables
	0 genotypes variables
Processing 1 interactions
-------------------------
Running 1 interactions using 14 processes...
	Finished Running 1 interactions
0 tests had an error
Completed Interaction Study for HI_CHOL

Completed association study
InteractionRegression
-------------------------
Binary Outcome (family = Binomial): 'HI_CHOL'
	7,059 occurrences of '0.0' coded as 0
	787 occurrences of '1.0' coded as 1
Using 7,846 of 8,591 observations
	745 are missing a value for the outcome variable
Regressing 3 variables
	1 binary variables
	1 categorical variables
	1 continuous variables
	0 genotypes variables
Processing 1 interactions
----------------

/Users/andrerico/Works/Temp/ge/venv/lib/python3.11/site-packages/clarite/modules/analyze/regression/glm_regression.py:142: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.data[self.outcome_variable].replace(categories, codes, inplace=True)
/Users/andrerico/Works/Temp/ge/venv/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The logit link alias is deprecated. Use Logit instead. The logit link alias will be removed after the 0.15.0 release.
  warnings.warn(
/Users/andrerico/Works/Temp/ge/ve

<h5>Test the nhanes dataset a specific interaction (using 'weight' as a continuous variable).</h5>
Not realistic, but good enough for a test.

---

In [60]:
# Prepare the data
nhanes_data = igem.load.from_csv(data_dir / "nhanes_data.csv", index_col=None)

df = igem.modify.colfilter(
    nhanes_data, only=["HI_CHOL", "RIAGENDR", "race", "agecat", "WTMEC2YR"]
)
df = igem.modify.make_binary(df, only=["HI_CHOL", "RIAGENDR"])
df = igem.modify.make_categorical(df, only=["race", "agecat"])

Loaded 8,591 observations of 7 variables
Running colfilter
--------------------------------------------------------------------------------
Keeping 5 of 7 variables:
	0 of 0 binary variables
	0 of 0 categorical variables
	4 of 6 continuous variables
	1 of 1 unknown variables
Running make_binary
--------------------------------------------------------------------------------
Set 2 of 5 variable(s) as binary, each with 8,591 observations
Running make_categorical
--------------------------------------------------------------------------------
Set 2 of 5 variable(s) as categorical, each with 8,591 observations


In [61]:
# Run Interaction without reporting betas
python_result_nobeta = igem.analyze.interaction_study(
    outcomes="HI_CHOL",
    covariates=["agecat", "RIAGENDR"],
    data=df,
    interactions=[("WTMEC2YR", "race")],
    report_betas=False,
)

# Run Interaction with reporting betas
python_result = igem.analyze.interaction_study(
    outcomes="HI_CHOL",
    covariates=["agecat", "RIAGENDR"],
    data=df,
    interactions=[("WTMEC2YR", "race")],
    report_betas=True,
)

InteractionRegression
-------------------------
Binary Outcome (family = Binomial): 'HI_CHOL'
	7,059 occurrences of '0.0' coded as 0
	787 occurrences of '1.0' coded as 1
Using 7,846 of 8,591 observations
	745 are missing a value for the outcome variable
Regressing 2 variables
	0 binary variables
	1 categorical variables
	1 continuous variables
	0 genotypes variables
Processing 1 interactions
-------------------------
Running 1 interactions using 14 processes...
	Finished Running 1 interactions
0 tests had an error
Completed Interaction Study for HI_CHOL

Completed association study
InteractionRegression
-------------------------
Binary Outcome (family = Binomial): 'HI_CHOL'
	7,059 occurrences of '0.0' coded as 0
	787 occurrences of '1.0' coded as 1
Using 7,846 of 8,591 observations
	745 are missing a value for the outcome variable
Regressing 2 variables
	0 binary variables
	1 categorical variables
	1 continuous variables
	0 genotypes variables
Processing 1 interactions
----------------

/Users/andrerico/Works/Temp/ge/venv/lib/python3.11/site-packages/clarite/modules/analyze/regression/glm_regression.py:142: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.data[self.outcome_variable].replace(categories, codes, inplace=True)
/Users/andrerico/Works/Temp/ge/venv/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The logit link alias is deprecated. Use Logit instead. The logit link alias will be removed after the 0.15.0 release.
  warnings.warn(
/Users/andrerico/Works/Temp/ge/ve

<h5>Test the nhanes dataset a Pairwise Interction</h5>

---

In [57]:
# Prepare the data
nhanes_data = igem.load.from_csv(data_dir / "nhanes_data.csv", index_col=None)

df = igem.modify.colfilter(
    nhanes_data, only=["HI_CHOL", "RIAGENDR", "race", "agecat"]
)
df = igem.modify.make_binary(df, only=["HI_CHOL", "RIAGENDR"])
df = igem.modify.make_categorical(df, only=["race", "agecat"])

Loaded 8,591 observations of 7 variables
Running colfilter
--------------------------------------------------------------------------------
Keeping 4 of 7 variables:
	0 of 0 binary variables
	0 of 0 categorical variables
	3 of 6 continuous variables
	1 of 1 unknown variables
Running make_binary
--------------------------------------------------------------------------------
Set 2 of 4 variable(s) as binary, each with 8,591 observations
Running make_categorical
--------------------------------------------------------------------------------
Set 2 of 4 variable(s) as categorical, each with 8,591 observations


In [58]:
# Run Interaction without reporting betas
python_result_nobeta = igem.analyze.interaction_study(
    outcomes="HI_CHOL",
    covariates=[],
    data=df,
    interactions=None,
    report_betas=False,
)

# Run Interaction with reporting betas
python_result = igem.analyze.interaction_study(
    outcomes="HI_CHOL",
    covariates=[],
    data=df,
    interactions=None,
    report_betas=True,
)

# Test Adding pvalues
igem.analyze.add_corrected_pvalues(python_result_nobeta, pvalue="LRT_pvalue")

igem.analyze.add_corrected_pvalues(python_result, pvalue="Full_Var1_Var2_beta")

igem.analyze.add_corrected_pvalues(python_result, pvalue="LRT_pvalue", groupby=["Term1", "Term2"])

InteractionRegression
-------------------------
Binary Outcome (family = Binomial): 'HI_CHOL'
	7,059 occurrences of '0.0' coded as 0
	787 occurrences of '1.0' coded as 1
Using 7,846 of 8,591 observations
	745 are missing a value for the outcome variable
Regressing 3 variables
	1 binary variables
	2 categorical variables
	0 continuous variables
	0 genotypes variables
Processing 3 interactions
-------------------------
Running 3 interactions using 14 processes...
	Finished Running 3 interactions
0 tests had an error
Completed Interaction Study for HI_CHOL

Completed association study
InteractionRegression
-------------------------
Binary Outcome (family = Binomial): 'HI_CHOL'
	7,059 occurrences of '0.0' coded as 0
	787 occurrences of '1.0' coded as 1
Using 7,846 of 8,591 observations
	745 are missing a value for the outcome variable
Regressing 3 variables
	1 binary variables
	2 categorical variables
	0 continuous variables
	0 genotypes variables
Processing 3 interactions
----------------

/Users/andrerico/Works/Temp/ge/venv/lib/python3.11/site-packages/clarite/modules/analyze/regression/glm_regression.py:142: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.data[self.outcome_variable].replace(categories, codes, inplace=True)
/Users/andrerico/Works/Temp/ge/venv/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The logit link alias is deprecated. Use Logit instead. The logit link alias will be removed after the 0.15.0 release.
  warnings.warn(
/Users/andrerico/Works/Temp/ge/ve

	Finished Running 3 interactions
0 tests had an error
Completed Interaction Study for HI_CHOL

Completed association study


<h4>Analyse Association Study</h4>

TODO

<h2>PLOT</h2>

TODO